# Phase 3 &mdash; Bronze Layer

This is the first stage of our Medallion pipeline. Its job is to take the Lending Club loan data exactly as it arrived and register it as a managed Delta table. No cleaning, no filtering, no type changes &mdash; that all happens in the Silver layer. Keeping Bronze untouched means we always have a faithful record of the source, and that any bug we introduce later can be traced back by re-reading the raw table.

### Input
The 2014&ndash;2017 Lending Club CSV produced by `src/01_data_loading.py`. We uploaded it through the Databricks Catalog UI, which automatically converted the upload into a managed Delta table at `workspace.default.optimized_data_14_17`.

### Output
`workspace.default.bronze_loans` &mdash; an identical copy of the uploaded table, renamed under our medallion convention.

### Why a sanitization step?
The pandas export that produced the CSV left an `Unnamed: 0` index column. Delta rejects column names containing spaces, colons, or other punctuation, so we run a one-pass rename that replaces any non-alphanumeric character with an underscore and lowercases every name. It is a *cosmetic* change &mdash; the row values are identical to the uploaded data &mdash; but it lets Delta accept the schema.

In [0]:
# A SparkSession named `spark` is already available on a Databricks cluster.
# The try/except keeps the notebook runnable outside Databricks too (e.g. in a
# local `pip install pyspark delta-spark` environment for development).
try:
    spark  # type: ignore[name-defined]
except NameError:
    from pyspark.sql import SparkSession
    spark = (
        SparkSession.builder.appName("phase3-bronze")
        .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .getOrCreate()
    )

print(f"Spark {spark.version} ready.")

Spark 4.1.0 ready.


In [0]:
# Source and destination tables. Update SOURCE_TABLE if the uploaded CSV landed
# somewhere other than workspace.default.
SOURCE_PATH = "/Volumes/workspace/default/raw_data/optimized_data_14_17.csv"
BRONZE_TABLE = "workspace.default.bronze_loans"

## Step 1 &mdash; Load the uploaded table

No transformations. Just read it into a Spark DataFrame and confirm the shape.

In [0]:
raw_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(SOURCE_PATH)
)

print(f"Rows: {raw_df.count():,}")
print(f"Columns: {len(raw_df.columns)}")
display(raw_df.limit(5))

Rows: 1,534,710
Columns: 142


Unnamed: 0,id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,url,purpose,title,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,last_fico_range_high,last_fico_range_low,collections_12_mths_ex_med,mths_since_last_major_derog,policy_code,application_type,annual_inc_joint,dti_joint,verification_status_joint,acc_now_delinq,tot_coll_amt,tot_cur_bal,open_acc_6m,open_act_il,open_il_12m,open_il_24m,mths_since_rcnt_il,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m,acc_open_past_24mths,avg_cur_bal,bc_open_to_buy,bc_util,chargeoff_within_12_mths,delinq_amnt,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,mths_since_recent_bc_dlq,mths_since_recent_inq,mths_since_recent_revol_delinq,num_accts_ever_120_pd,num_actv_bc_tl,num_actv_rev_tl,num_bc_sats,num_bc_tl,num_il_tl,num_op_rev_tl,num_rev_accts,num_rev_tl_bal_gt_0,num_sats,num_tl_120dpd_2m,num_tl_30dpd,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,revol_bal_joint,sec_app_fico_range_low,sec_app_fico_range_high,sec_app_earliest_cr_line,sec_app_inq_last_6mths,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_chargeoff_within_12_mths,sec_app_collections_12_mths_ex_med,hardship_flag,hardship_type,hardship_reason,hardship_status,deferral_term,hardship_amount,hardship_start_date,hardship_end_date,payment_plan_start_date,hardship_length,hardship_dpd,hardship_loan_status,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount,debt_settlement_flag
223270,10735962,24000,24000,24000.0,60 months,24.50%,697.42,F,F3,CI Engineer,10+ years,MORTGAGE,100000.0,Verified,2014-01-01,Fully Paid,n,https://lendingclub.com/browse/loanDetail.action?loan_id=10735962,debt_consolidation,Debt consolidation,840xx,UT,17.52,0.0,Oct-1981,680,684,1.0,29.0,null,12.0,0,3243,81.1%,38,f,0.0,0.0,38198.7073634409,38198.71,24000.0,14198.71,0.0,0.0,0.0,Feb-2017,55.32,null,Jul-2019,524,520,0,29.0,1,Individual,null,null,null,0,0.0,368061.0,null,null,null,null,null,null,null,null,null,null,null,4000.0,null,null,null,8.0,33460.0,null,null,1.0,0,124.0,387.0,8.0,2.0,6.0,166.0,39.0,6.0,39.0,5.0,0.0,3.0,1.0,6.0,17.0,4.0,15.0,3.0,12.0,0.0,0.0,0.0,4.0,76.3,null,0.0,0,385238.0,57920.0,0.0,70738.0,null,null,null,null,null,null,null,null,null,null,null,null,N,null,null,null,null,null,null,null,null,null,null,null,null,null,null,N
222285,10066442,15000,15000,14950.0,36 months,10.99%,491.01,B,B2,Administrative Assistant,10+ years,RENT,44500.0,Not Verified,2014-01-01,Fully Paid,n,https://lendingclub.com/browse/loanDetail.action?loan_id=10066442,debt_consolidation,Personal Loan,857xx,AZ,10.18,0.0,May-1997,700,704,0.0,36.0,null,15.0,0,6950,23.8%,36,f,0.0,0.0,16368.050002873,16313.49,15000.0,1368.05,0.0,0.0,0.0,Jan-2015,11457.95,null,Nov-2018,819,815,0,36.0,1,Individual,null,null,null,0,0.0,10449.0,null,null,null,null,null,null,null,null,null,null,null,29200.0,null,null,null,3.0,871.0,16928.0,24.4,0.0,0,139.0,199.0,7.0,3.0,2.0,7.0,null,24.0,null,1.0,4.0,7.0,9.0,13.0,9.0,14.0,25.0,7.0,15.0,0.0,0.0,0.0,2.0,97.2,16.7,0.0,0,33140.0,10449.0,22400.0,3940.0,null,null,null,null,null,null,null,null,null,null,null,null,N,null,null,null,null,null,null,null,null,null,null,null,null,null,null,N
222284,5716249,24000,24000,23850.0,36 m

## Step 2 &mdash; Sanity-check with a few rows and the schema

In [0]:
raw_df.limit(5).toPandas()

,Unnamed: 0,id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,url,purpose,title,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,...,num_tl_120dpd_2m,num_tl_30dpd,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,revol_bal_joint,sec_app_fico_range_low,sec_app_fico_range_high,sec_app_earliest_cr_line,sec_app_inq_last_6mths,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_chargeoff_within_12_mths,sec_app_collections_12_mths_ex_med,hardship_flag,hardship_type,hardship_reason,hardship_status,deferral_term,hardship_amount,hardship_start_date,hardship_end_date,payment_plan_start_date,hardship_length,hardship_dpd,hardship_loan_status,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount,debt_settlement_flag
0,223270,10735962,24000,24000,24000.0,60 months,24.50%,697.42,F,F3,CI Engineer,10+ years,MORTGAGE,100000.0,Verified,2014-01-01,Fully Paid,n,https://lendingclub.com/browse/loanDetail.acti...,debt_consolidation,Debt consolidation,840xx,UT,17.52,0.0,Oct-1981,680,684,1.0,29.0,NaN,12.0,0,3243,81.1%,38,f,0.0,0.0,38198.707363,...,0.0,0.0,0.0,4.0,76.3,NaN,0.0,0,385238.0,57920.0,0.0,70738.0,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,None,None,None,NaN,NaN,None,None,None,NaN,NaN,None,NaN,NaN,NaN,N
1,222285,10066442,15000,15000,14950.0,36 months,10.99%,491.01,B,B2,Administrative Assistant,10+ years,RENT,44500.0,Not Verified,2014-01-01,Fully Paid,n,https://lendingclub.com/browse/loanDetail.acti...,debt_consolidation,Personal Loan,857xx,AZ,10.18,0.0,May-1997,700,704,0.0,36.0,NaN,15.0,0,6950,23.8%,36,f,0.0,0.0,16368.050003,...,0.0,0.0,0.0,2.0,97.2,16.7,0.0,0,33140.0,10449.0,22400.0,3940.0,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,None,None,None,NaN,NaN,None,None,None,NaN,NaN,None,NaN,NaN,NaN,N
2,222284,5716249,24000,24000,23850.0,36 months,10.99%,785.62,B,B2,Project Manager,7 years,RENT,80000.0,Source Verified,2014-01-01,Fully Paid,n,https://lendingclub.com/browse/loanDetail.acti...,debt_consolidation,Debt consolidation,113xx,NY,10.08,0.0,Mar-1996,705,709,1.0,NaN,NaN,12.0,0,22093,53%,21,f,0.0,0.0,27508.650045,...,0.0,0.0,0.0,1.0,100.0,60.0,0.0,0,58139.0,31712.0,29400.0,16439.0,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,None,None,None,NaN,NaN,None,None,None,NaN,NaN,None,NaN,NaN,NaN,N
3,232839,10161312,8000,8000,8000.0,36 months,12.85%,268.98,B,B4,Accountability Project Manager,10+ years,RENT,62000.0,Not Verified,2014-01-01,Fully Paid,n,https://lendingclub.com/browse/loanDetail.acti...,debt_consolidation,Amazing Opportunity for Consolidation :),980xx,WA,20.64,2.0,Nov-2001,675,679,0.0,14.0,NaN,10.0,0,22414,78.4%,20,w,0.0,0.0,8947.910000,...,0.0,0.0,0.0,0.0,90.0,40.0,0.0,0,67295.0,43084.0,11600.0,38695.0,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,None,None,None,NaN,NaN,None,None,None,NaN,NaN,None,NaN,NaN,NaN,N
4,232840,10177692,12000,12000,12000.0,36 months,13.98%,410.02,C,C1,Systems Admin,2 years,MORTGAGE,69000.0,Not Verified,2014-01-01,Fully Paid,n,https://lendingclub.com/browse/loanDetail.acti...,debt_consolidation,Debt consolidation,851xx,AZ,10.59,1.0,Feb-2002,670,674,1.0,14.0,NaN,6.0,0,17917,64.4%,18,f,0.0,0.0,14274.071158,...,0.0,0.0,0.0,1.0,94.4,50.0,0.0,0,178893.0,22647.0,13000.0,5690.0,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,None,None,None,NaN,NaN,None,None,None,NaN,NaN,None,NaN,NaN,NaN,N


In [0]:
raw_df.printSchema()

root
 |-- Unnamed: 0: integer (nullable = true)
 |-- id: integer (nullable = true)
 |-- loan_amnt: integer (nullable = true)
 |-- funded_amnt: integer (nullable = true)
 |-- funded_amnt_inv: double (nullable = true)
 |-- term: string (nullable = true)
 |-- int_rate: string (nullable = true)
 |-- installment: double (nullable = true)
 |-- grade: string (nullable = true)
 |-- sub_grade: string (nullable = true)
 |-- emp_title: string (nullable = true)
 |-- emp_length: string (nullable = true)
 |-- home_ownership: string (nullable = true)
 |-- annual_inc: string (nullable = true)
 |-- verification_status: string (nullable = true)
 |-- issue_d: string (nullable = true)
 |-- loan_status: string (nullable = true)
 |-- pymnt_plan: string (nullable = true)
 |-- url: string (nullable = true)
 |-- purpose: string (nullable = true)
 |-- title: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- addr_state: string (nullable = true)
 |-- dti: string (nullable = true)
 |-- delinq_2

## Step 3 &mdash; Sanitize column names

Replace any character outside `[A-Za-z0-9_]` with `_` and lowercase the result. In our data this only renames `Unnamed: 0` (a pandas index artifact) to `unnamed_0`, but the same rule safely handles anything else.

In [0]:
import re

def sanitize(name):
    cleaned = re.sub(r"[^A-Za-z0-9_]+", "_", name).strip("_").lower()
    return cleaned or "col"

renamed_df = raw_df
changed = []
for old in raw_df.columns:
    new = sanitize(old)
    if new != old:
        renamed_df = renamed_df.withColumnRenamed(old, new)
        changed.append((old, new))

print(f"Renamed {len(changed)} column(s):")
for old, new in changed:
    print(f"  {old!r:<20} -> {new!r}")

Renamed 1 column(s):
  'Unnamed: 0'         -> 'unnamed_0'


## Step 4 &mdash; Write the Bronze Delta table

`mode("overwrite")` makes re-runs idempotent &mdash; the table ends in the same state whether this is the first run or the tenth.

In [0]:
(
    renamed_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_TABLE)
)

print(f"Wrote Delta table: {BRONZE_TABLE}")

Wrote Delta table: workspace.default.bronze_loans


## Step 5 &mdash; Verify

In [0]:
spark.sql(f"SELECT COUNT(*) AS row_count FROM {BRONZE_TABLE}").show()
spark.sql(f"DESCRIBE DETAIL {BRONZE_TABLE}") \
    .select("name", "location", "numFiles", "sizeInBytes") \
    .show(truncate=False)

+---------+
|row_count|
+---------+
|  1534710|
+---------+

+------------------------------+--------+--------+-----------+
|name                          |location|numFiles|sizeInBytes|
+------------------------------+--------+--------+-----------+
|workspace.default.bronze_loans|        |8       |182784846  |
+------------------------------+--------+--------+-----------+

